In [14]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

In [15]:
def check_data_type(y_true, prot_attr):

    if isinstance(y_true, pd.Series):
        y_true = y_true.to_numpy()
    if isinstance(prot_attr, pd.Series):
        prot_attr = prot_attr.to_numpy()  

    return y_true, prot_attr

def get_neg_label(y_true, pos_label):

    unique_y_true = np.unique(y_true)
    if len(unique_y_true) == 1:
        raise ValueError("y_true contains only one unique value, unable to determine negative label.")
    else:
        neg_label_list = [x for x in unique_y_true if x != pos_label]
        neg_label = neg_label_list[0] if len(unique_y_true) == 2 else neg_label_list
    
    return neg_label

def get_unpriv_group(prot_attr, priv_group):

    unique_prot_attr = np.unique(prot_attr)
    if len(unique_prot_attr) == 1:
        raise ValueError("prot_attr contains only one unique value, unable to determine unprivileged group.")  
    else:
        unpriv_group_list = [x for x in unique_prot_attr if x != priv_group]
        unpriv_group = unpriv_group_list[0] if len(unique_prot_attr) == 2 else unpriv_group_list

    return unpriv_group

In [40]:
def positive_predicted_value_unpriv(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    unpriv_group = get_unpriv_group(prot_attr, priv_group)

    true_positive_idx = np.where(y_true == pos_label)[0]  
    pred_positive_idx = np.where(y_pred == pos_label)[0]  
    unpriv_idx = np.where(prot_attr == unpriv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in unpriv_group)[0]
    pred_positive_unpriv_idx = np.intersect1d(pred_positive_idx, unpriv_idx)
    pred_positive_unpriv_true_pos_idx = np.intersect1d(pred_positive_unpriv_idx, true_positive_idx)
    PPV_unpriv_num = len(pred_positive_unpriv_true_pos_idx)
    print('PPV_unpriv_num', PPV_unpriv_num)
    PPV_unpriv_den = len(pred_positive_unpriv_idx)
    print('PPV_unpriv_den', PPV_unpriv_den)     
    if PPV_unpriv_den == 0:
        PPV_unpriv_den = 1e-5
    PPV_unpriv = PPV_unpriv_num / PPV_unpriv_den
    print('PPV_unpriv', PPV_unpriv)

    return PPV_unpriv

def positive_predicted_value_priv(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)

    true_positive_idx = np.where(y_true == pos_label)[0]  
    pred_positive_idx = np.where(y_pred == pos_label)[0]  
    priv_idx = np.where(prot_attr == priv_group)[0] 
    pred_positive_priv_idx = np.intersect1d(pred_positive_idx, priv_idx)
    pred_positive_priv_true_pos_idx = np.intersect1d(pred_positive_priv_idx, true_positive_idx)
    PPV_priv_num = len(pred_positive_priv_true_pos_idx)
    PPV_priv_den = len(pred_positive_priv_idx)
    if PPV_priv_den == 0:
        PPV_priv_den = 1e-5
    PPV_priv = PPV_priv_num / PPV_priv_den

    return PPV_priv

In [41]:
def positive_predicted_value_ratio(y_true, y_pred, prot_attr, priv_group, pos_label):

    PPV_unpriv = positive_predicted_value_unpriv(y_true, y_pred, prot_attr, priv_group, pos_label)
    PPV_priv = positive_predicted_value_priv(y_true, y_pred, prot_attr, priv_group, pos_label) 
    
    if PPV_priv == 0:
        PPV_priv = 1e-5
    
    PPV_ratio = PPV_unpriv / PPV_priv   

    return PPV_ratio

In [42]:
fairness_metrics = {'positive_predicted_value_ratio': positive_predicted_value_ratio}

In [43]:
def cross_val_score_fairness(estimator, X, y, sens_variable_name, priv_group, pos_label, scoring, cv):
    
    # X must be a Pandas DataFrame
    if not isinstance(X, pd.DataFrame):
        raise TypeError('X must be a Pandas DataFrame')
    if isinstance(y, pd.Series):
        y = y.to_numpy()

    metric_iters = []
    # Split the data into training and validation sets 
    for train_index, val_index in cv.split(X, y): 
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        Y_train, Y_val = y[train_index], y[val_index]
        A_val = X_val[sens_variable_name] # sensitive variable in val set

        # Train logistic regression model
        estimator.fit(X_train, Y_train)

        # Predict on validation set
        Y_val_hat = estimator.predict(X_val)

        # Calculate fairness metrics for each iteration of the cross-validation process
        metric_iters.append(fairness_metrics[scoring](y_true=Y_val, y_pred=Y_val_hat, prot_attr=A_val,
                                                    priv_group=priv_group, pos_label=pos_label))

            
    # Compute the average of the metric along the iterations
    final_metric = np.mean(metric_iters)

    return final_metric, metric_iters

###############################################################################################################################

class RandomizedSearchCVFairness:
    
    def __init__(self, estimator, param_distributions, 
                 fairness_scoring, predictive_scoring, objective, 
                 fairness_scoring_direction, predictive_scoring_direction, 
                 fairness_weight, predictive_weight, 
                 cv, n_iter, random_state, sens_variable_name, priv_group, pos_label):

        self.estimator = estimator
        self.param_distributions = param_distributions
        self.fairness_scoring = fairness_scoring
        self.predictive_scoring = predictive_scoring
        self.objective = objective # ['fairness', 'predictive']
        self.fairness_scoring_direction = fairness_scoring_direction # ['maximize', ' minimize']
        self.predictive_scoring_direction = predictive_scoring_direction # ['maximize', ' minimize']
        self.fairness_weight = fairness_weight
        self.predictive_weight = predictive_weight
        self.cv = cv 
        self.n_iter = n_iter
        self.random_state = random_state
        self.sens_variable_name = sens_variable_name
        self.priv_group = priv_group
        self.pos_label = pos_label
        self.results_ = []

    def _random_param_sample(self):
        """Randomly sample a parameter combination from the distributions."""
        random_params = {key: random.choice(val) for key, val in self.param_distributions.items()}
        return random_params
    
    def fit(self, X, y):
        
        random.seed(self.random_state)
        for iter in range(self.n_iter):
            print('GridSearch Iter:', iter)
            try:
                random_params = self._random_param_sample()
                self.estimator.set_params(**random_params)

                fairness_final_metric, _ = cross_val_score_fairness(estimator=self.estimator, X=X, y=y, 
                                                                    sens_variable_name=self.sens_variable_name, 
                                                                    priv_group=self.priv_group,
                                                                    pos_label=self.pos_label, 
                                                                    scoring=self.fairness_scoring, cv=self.cv)  
                predictive_metric_iters = cross_val_score(estimator=self.estimator, X=X, y=y, 
                                                          scoring=self.predictive_scoring, cv=self.cv)
                
                predictive_final_metric = np.mean(predictive_metric_iters)
                 
                self.results_.append({'params': random_params, 
                                      'predictive-score': predictive_final_metric, 
                                      'fairness-score': fairness_final_metric})
                
            except Exception as e:
                print('RandomizedSearchCVFairness Exception:', e)
            
            predictive_scores = [self.results_[i][f'predictive-score'] for i in range(len(self.results_))]
            fairness_scores = [self.results_[i][f'fairness-score'] for i in range(len(self.results_))]
            
            # Building the combined score
            scaler = MinMaxScaler(feature_range=(0,1))            
            predictive_scores_normalized = scaler.fit_transform(np.array(predictive_scores).reshape(-1, 1)).flatten()
            fairness_scores_normalized = scaler.fit_transform(np.array(fairness_scores).reshape(-1, 1)).flatten()
            if self.predictive_scoring_direction == 'minimize':
                predictive_scores_normalized = 1 - predictive_scores_normalized
            if self.fairness_scoring_direction == 'minimize':
                fairness_scores_normalized = 1 - fairness_scores_normalized
            if  self.predictive_weight + self.fairness_weight == 1:
                combined_scores = predictive_scores_normalized * self.predictive_weight + fairness_scores_normalized * self.fairness_weight
            else:
                raise ValueError("The sum of predictive_weight and fairness_weight must be 1.")
            for i in range(len(self.results_)):
                self.results_[i]['combined-score'] = combined_scores[i]
            
            # Optimizing the parameters according to the objective and the scores
            # Obtaining the best params and score, and building a data-frame with the results
            score_list = [self.results_[i][f'{self.objective}-score'] for i in range(len(self.results_))]
            self.cv_results_ = pd.DataFrame(self.results_)

            scoring_direction_map = {'combined': ('maximize', False),
                                     'fairness': (self.fairness_scoring_direction, None),
                                     'predictive': (self.predictive_scoring_direction, None)
                                    }
            scoring_direction, ascending_value = scoring_direction_map[self.objective]
            opt_function = np.argmax if scoring_direction == 'maximize' else np.argmin
            ascending_value = False if scoring_direction == 'maximize' else True 
            best_score_idx = opt_function(score_list)
            self.cv_results_ = self.cv_results_.sort_values(by=f'{self.objective}-score', ascending=ascending_value)
            self.best_params_ = self.results_[best_score_idx]['params']
            self.best_score_ = self.results_[best_score_idx][f'{self.objective}-score']


---

In [20]:
from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

from sklearn.model_selection import train_test_split, StratifiedKFold

from aif360.datasets import GermanDataset

from PyMachineLearning.models import LogisticRegressionThreshold

pd.set_option('display.max_colwidth', None)

In [8]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [9]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
# The sensitive variable/s must index the df in order to avoid problems with some aif360 processors (PostProcessingMetaEstimator)
german_data_pd.index = german_data_pd[sens_variable_name] 

In [10]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=111)
log_reg_estimator = LogisticRegressionThreshold(solver='liblinear', random_state=123, max_iter=200)

In [12]:
param_grid = {
    'penalty': ['l1', 'l2'], 
    'C': [0.01, 0.1, 1, 10, 30, 50, 75, 100],  # Regularization strength
    'class_weight': ['balanced', None],
    'threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

In [44]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                        fairness_scoring='positive_predicted_value_ratio', 
                                                        predictive_scoring='balanced_accuracy',
                                                        objective='combined', 
                                                        fairness_scoring_direction='minimize',
                                                        predictive_scoring_direction='maximize',
                                                        fairness_weight=0.5, predictive_weight=0.5,
                                                        cv=inner, n_iter=20, random_state=123,
                                                        sens_variable_name=sens_variable_name, 
                                                        priv_group=sens_privileged_groups[0][sens_variable_name],
                                                        pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

fairness_random_search.cv_results_.head(3)

GridSearch Iter: 0
PPV_unpriv_num 6
PPV_unpriv_den 7
PPV_unpriv 0.8571428571428571
PPV_unpriv_num 0
PPV_unpriv_den 0
PPV_unpriv 0.0
PPV_unpriv_num 1
PPV_unpriv_den 1
PPV_unpriv 1.0
PPV_unpriv_num 4
PPV_unpriv_den 4
PPV_unpriv 1.0
PPV_unpriv_num 2
PPV_unpriv_den 4
PPV_unpriv 0.5
GridSearch Iter: 1
PPV_unpriv_num 5
PPV_unpriv_den 5
PPV_unpriv 1.0
PPV_unpriv_num 0
PPV_unpriv_den 0
PPV_unpriv 0.0
PPV_unpriv_num 0
PPV_unpriv_den 0
PPV_unpriv 0.0
PPV_unpriv_num 2
PPV_unpriv_den 2
PPV_unpriv 1.0
PPV_unpriv_num 1
PPV_unpriv_den 1
PPV_unpriv 1.0
GridSearch Iter: 2
PPV_unpriv_num 12
PPV_unpriv_den 19
PPV_unpriv 0.631578947368421
PPV_unpriv_num 5
PPV_unpriv_den 7
PPV_unpriv 0.7142857142857143
PPV_unpriv_num 8
PPV_unpriv_den 10
PPV_unpriv 0.8
PPV_unpriv_num 9
PPV_unpriv_den 9
PPV_unpriv 1.0
PPV_unpriv_num 7
PPV_unpriv_den 11
PPV_unpriv 0.6363636363636364
GridSearch Iter: 3
PPV_unpriv_num 14
PPV_unpriv_den 23
PPV_unpriv 0.6086956521739131
PPV_unpriv_num 9
PPV_unpriv_den 13
PPV_unpriv 0.692307692307

,params,predictive-score,fairness-score,combined-score
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced', 'threshold': 0.8}",0.672269,0.713974,0.505951
9,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.9}",0.671148,0.713244,0.503910
10,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.8}",0.670308,0.713974,0.501724
